# Investigating Netflix Movies

This notebook performs exploratory data analysis on the Netflix titles dataset with a movie-only focus. Instead of stopping at a single scatter plot, the analysis checks data quality, explores catalog composition, studies duration patterns, and compares genres, countries, and release periods.

## Guiding Questions

- What does the dataset contain, and how complete is it?
- How large is the movie catalog compared with TV shows?
- Are movie durations actually getting shorter over time?
- Which genres dominate the Netflix movie catalog?
- Which countries appear most often in the movie data?
- What changes when we zoom in on the 1990s?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
netflix_df = pd.read_csv("netflix_data.csv")
netflix_df.head()

## 1. Dataset Overview

The dataset contains **7,787** Netflix titles split between movies and TV shows. For this project, the main analysis focuses on the **5,377 movie records**.

In [ ]:
netflix_df.shape, netflix_df['type'].value_counts()

In [ ]:
missing_summary = netflix_df.isna().sum().sort_values(ascending=False)
missing_summary

### Missingness Notes

- `director` has the most missing values, which is common in content datasets with anthology titles or incomplete metadata.
- `cast` and `country` also have notable gaps.
- Core analytical fields like `type`, `release_year`, `duration`, and `genre` are complete, which makes the movie-level EDA workable.

## 2. Creating the Movie-Only Dataset

The original dataset mixes movies and TV shows. Since the research question is about **movie duration**, TV shows are excluded before the rest of the analysis.

In [ ]:
movies = netflix_df[netflix_df['type'] == 'Movie'].copy()
movies['primary_country'] = movies['country'].fillna('Unknown').str.split(',').str[0].str.strip()
movies['decade'] = (movies['release_year'] // 10) * 10
movies[['title', 'genre', 'release_year', 'duration', 'primary_country']].head()

In [ ]:
movies[['release_year', 'duration']].describe()

### Movie Snapshot

- Average movie duration: **99.31 minutes**
- Median movie duration: **98 minutes**
- 75% of movie durations are **114 minutes or less**
- The catalog is heavily concentrated in the **2010s**

## 3. Catalog Composition

Before looking at durations, it helps to see the overall content split between movies and TV shows.

![Netflix Titles by Content Type](plots/content_type_split.png)

Movies make up the larger share of titles in the dataset, which supports narrowing the analysis to film entries.

## 4. Release Trend Over Time

A useful first EDA step is checking how the catalog is distributed across release years.

In [ ]:
movies.groupby('release_year').size().tail(15)

![Number of Netflix Movies by Release Year](plots/movies_by_release_year.png)

The movie catalog is highly skewed toward more recent releases, especially from the 2010s onward. That means any duration conclusion needs to be read in the context of a dataset dominated by newer titles.

## 5. Duration Distribution

Before testing the 'movies are getting shorter' idea, it helps to understand the basic distribution of runtimes.

In [ ]:
movies['duration'].quantile([0.10, 0.25, 0.50, 0.75, 0.90])

![Distribution of Netflix Movie Durations](plots/duration_distribution.png)

Most movie durations cluster around standard feature-length values, roughly from the mid-80s to the low-110s. Very short runtimes exist, but they are relatively uncommon in the full movie catalog.

## 6. Duration Over Time

This is the core visual for the original project question.

In [ ]:
movies.groupby('release_year')['duration'].mean().tail(15).round(2)

![Netflix Movie Duration by Release Year](plots/movie_duration_by_year.png)

The scatter plot shows a broad spread of runtimes across years rather than a simple straight-line decline. There is evidence that more recent titles are somewhat shorter on average than older decades, but the pattern is gradual and noisy rather than dramatic.

## 7. Average Duration by Decade

Looking at decade-level averages gives a cleaner view than the year-by-year scatter.

In [ ]:
movies.groupby('decade')['duration'].agg(['count', 'mean', 'median']).round(2)

![Average Movie Duration by Decade](plots/average_duration_by_decade.png)

The decade view shows that the 2010s and 2020s trend shorter than many earlier decades. That supports a **mild shortening pattern**, but not the simplistic claim that all movies are suddenly short. The catalog still contains many standard-length films.

## 8. Genre Analysis

Genre mix matters because runtime expectations differ across categories like stand-up, documentaries, children's films, and dramas.

In [ ]:
movies['genre'].value_counts().head(10)

In [ ]:
movies.groupby('genre')['duration'].agg(['count', 'mean']).sort_values('count', ascending=False).head(10).round(2)

![Top 8 Movie Genres on Netflix](plots/top_genres.png)

The catalog is led by **Dramas**, **Comedies**, and **Documentaries**. Genre also helps explain runtime variation: documentaries and stand-up titles are noticeably shorter on average than action or drama films.

## 9. Country Analysis

Country representation can also shape catalog patterns, so a quick geographic check is useful.

In [ ]:
movies['primary_country'].value_counts().head(10)

![Top 8 Countries for Netflix Movies](plots/top_countries.png)

The **United States** dominates the dataset, followed by **India** and the **United Kingdom**. This suggests that any high-level pattern in the movie catalog is influenced heavily by a few countries with large title counts.

## 10. Zooming in on the 1990s

The original project often focused on the 1990s, so this section isolates that decade for a more specific look.

In [ ]:
movies_90s = movies[(movies['release_year'] >= 1990) & (movies['release_year'] < 2000)].copy()
movies_90s[['title', 'release_year', 'duration', 'genre']].sort_values('duration').head(10)

### 1990s Findings

- Number of 1990s movies: **194**
- Average 1990s duration: **113.92 minutes**
- Median 1990s duration: **108 minutes**
- Many of the shortest 1990s titles are stand-up specials or documentaries

That means very short runtimes in the 1990s slice are real, but they are not representative of the whole decade.

## Conclusion

A fuller EDA view leads to a more balanced conclusion than the original one-line answer:

1. Movie durations are **not collapsing suddenly**, but newer decades do show somewhat shorter average runtimes than several older decades.
2. The Netflix movie catalog is dominated by the **2010s**, so recent content strongly shapes the overall pattern.
3. Runtime differences are also tied to **genre mix**. Stand-up, children's titles, and documentaries pull average durations downward.
4. The catalog is geographically concentrated, with the **United States** and **India** contributing a large share of titles.

So the stronger takeaway is: **movie duration patterns vary by time period, genre, and catalog mix; they are not explained well by a single yes/no answer.**